# Phase6 DAEAC MCC Kaggle Driver

This notebook clones the repo, copies preprocessed `.npz` files and `daeac_base_best.pt`, runs leakage checks, trains MCC variants, evaluates, visualizes, and zips outputs. Training logic lives in repo scripts only.

In [ ]:
import os, shutil, glob, pathlib
REPO_URL = 'https://github.com/your-user/Best-thesis-in-council.git'
BRANCH = 'another-branch'
WORK = pathlib.Path('/kaggle/working')
REPO = WORK / 'Best-thesis-in-council'
if not REPO.exists():
    !git clone --branch {BRANCH} {REPO_URL} {REPO}
ECG = REPO / 'ecg_thesis'
os.chdir(ECG)
print('cwd =', os.getcwd())

In [ ]:
candidates = [p.parent for p in pathlib.Path('/kaggle/input').glob('**/record_split_manifest.json')]
assert len(candidates) == 1, f'Expected one record-split bundle, found {candidates}'
DATA_SRC = candidates[0]
DATA_DST = ECG / 'data/processed/phase6_daeac_record_splits'
DATA_DST.mkdir(parents=True, exist_ok=True)
for src in DATA_SRC.glob('*.npz'):
    shutil.copy2(src, DATA_DST / src.name)
print(sorted(p.name for p in DATA_DST.glob('*.npz')))
ckpts = {name: glob.glob(f'/kaggle/input/**/{name}', recursive=True) for name in ['daeac_base_best.pt', 'daeac_base_mitbih_best.pt']}
assert all(len(v) == 1 for v in ckpts.values()), f'Attach exactly one checkpoint per source: {ckpts}'
DS1_BASE_CKPT, MITBIH_BASE_CKPT = ckpts['daeac_base_best.pt'][0], ckpts['daeac_base_mitbih_best.pt'][0]
PAIRS = ['ds1_ds2', 'ds1_incart', 'ds1_svdb', 'mitbih_incart', 'mitbih_svdb']

In [ ]:
!python scripts/check_repo.py
!python scripts/daeac/03_validate_kaggle_bundle.py --bundle {DATA_SRC}
!python -m unittest tests.test_dan_mkmmd
!python scripts/phase6_daeac_paper/00_validate_data.py --config configs/phase6_daeac_mcc.yaml

The strict protocol check accepts the paper's intentional transductive overlap: DS2 first-five-minute inputs are a subset of full-DS2 evaluation, while target labels remain hidden during training.

In [ ]:
!python scripts/phase6_daeac_mcc/00_check_protocol.py --config configs/phase6_daeac_mcc.yaml --strict

In [ ]:
# Smoke runs
!python scripts/phase6_daeac_mcc/01_train.py --config configs/phase6_daeac_mcc.yaml --domain-pair ds1_ds2 --init-checkpoint {DS1_BASE_CKPT} --epochs 1 --max-source-samples 512 --max-target-samples 512 --max-val-samples 512 --checkpoint-prefix daeac_mcc_smoke

In [ ]:
# Full MCC runs
for pair in PAIRS:
    checkpoint = MITBIH_BASE_CKPT if pair.startswith('mitbih_') else DS1_BASE_CKPT
    if pathlib.Path(f'outputs/phase6_daeac_mcc_{pair}/checkpoints/daeac_mcc_{pair}_best.pt').exists(): print('resume: skip', pair); continue
    !python scripts/phase6_daeac_mcc/01_train.py --config configs/phase6_daeac_mcc.yaml --domain-pair {pair} --init-checkpoint {checkpoint}

In [ ]:
for pair in PAIRS:
    prefix = f'daeac_mcc_{pair}'
    ckpt = f'outputs/phase6_daeac_mcc_{pair}/checkpoints/{prefix}_best.pt'
    !python scripts/phase6_daeac_paper/03_eval.py --config configs/phase6_daeac_mcc.yaml --domain-pair {pair} --checkpoint {ckpt} --method-name {prefix}_best_v --dataset target

In [ ]:
!zip -r /kaggle/working/phase6_daeac_mcc_outputs.zip outputs/phase6_daeac_mcc_* outputs/phase6_daeac_paper_*